In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

In [3]:
# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [4]:
# 2. Feature Engineering: Create Color Indices
# We create these for BOTH train and test to ensure they match
for df in [train_df, test_df]:
    df['u_minus_g'] = df['u'] - df['g']
    df['g_minus_r'] = df['g'] - df['r']
    df['r_minus_i'] = df['r'] - df['i']
    df['i_minus_z'] = df['i'] - df['z']
    # A common astronomical feature: total magnitude
    df['total_mag'] = df['u'] + df['g'] + df['r'] + df['i'] + df['z']

In [5]:
# 3. Prepare Features and Target
X_train = train_df.drop(['id', 'class'], axis=1)
y_train = train_df['class']
X_test = test_df.drop(['id'], axis=1)

In [6]:
# 4. Encode Categorical Variables
cat_cols = ['spectral_type', 'galaxy_population']
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

In [9]:
# 5. Local Cross-Validation (to trust our new score)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("--- Starting Upgraded 5-Fold CV with LightGBM ---")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # LightGBM Classifier
    # scale_pos_weight helps with imbalance, but we'll use class_weight via params if needed
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        random_state=42,
        verbose=-1, # Silence training output
        n_jobs=-1
    )
    
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    cv_scores.append(score)
    print(f"Fold {fold+1} Balanced Accuracy: {score:.4f}")

mean_cv = sum(cv_scores) / len(cv_scores)
print(f"\n*** New Mean Local CV Balanced Accuracy: {mean_cv:.4f} ***")

--- Starting Upgraded 5-Fold CV with LightGBM ---
Fold 1 Balanced Accuracy: 0.9518
Fold 2 Balanced Accuracy: 0.9559
Fold 3 Balanced Accuracy: 0.9530
Fold 4 Balanced Accuracy: 0.9547
Fold 5 Balanced Accuracy: 0.9508

*** New Mean Local CV Balanced Accuracy: 0.9532 ***


In [10]:
# 6. Train Final Model and Submit
final_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1, n_jobs=-1)
final_model.fit(X_train, y_train)

test_preds = final_model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'class': test_preds
})

submission.to_csv('submission_v2.csv', index=False)
print("Success! submission_v2.csv is ready.")

Success! submission_v2.csv is ready.
